# Week 4 Lab. Training a VAE
**Deep Generative Models · GDS0058-1 · Practice, not graded**

Open this notebook from the link in the LMS lab notice, click **File > Save a copy in Drive** first, and work in your own copy. Nothing is submitted.

**How today's lab runs.** Not code first. Four steps, in this order.

| | | |
|---|---|---|
| 1 | **Predict** | Answer three questions in one line each, before any code runs. Being wrong is fine and expected. |
| 2 | **Run** | Execute the cells as given. Nothing to fill in, nothing to debug. |
| 3 | **Observe** | Write down what you actually see, not what you expected to see. |
| 4 | **Explain** | Say why it happened, in terms of q(z given x) and p(z). Only after that do you write code. |

*오늘 실습은 코드 빈칸부터 채우지 않습니다. 먼저 예측하고, 주어진 코드를 실행하고, 무엇이 보이는지 적고, 왜 그런지 설명한 다음에 코드를 씁니다. 예측이 틀려도 괜찮습니다. 틀린 예측을 적어 두었다가 결과와 비교하는 것이 이 방식의 핵심입니다.*

**Core path (55 min):** §0 setup 5 min → §1 predict 5 min → §2 run and observe 13 min → §3 your two lines 15 min → §4 the KL term on and off 12 min → §5 wrap-up 5 min. Optional sections follow §5. The class slot is 17:00 to 18:00, with a short briefing first. 수업 시간은 17:00~18:00 이고 앞에 짧은 안내가 있습니다.

**There are exactly two blanks today**, both in §3. Everything else is marked "run as is" and you are not expected to read it line by line.

Week 3 kept the model fixed and adjusted a guess q for one observation by hand. This week a neural network produces q(z given x) for every image, and the decoder is trained at the same time. Training runs three times in this notebook, about 30 to 60 seconds each.

> 요약: 지난주에는 모형을 고정하고 관측 하나에 대한 q를 손으로 조정했습니다. 이번 주에는 인코더가 이미지마다 q(z|x)를 만들어 주고, 디코더를 함께 학습합니다. 빈칸은 §3의 두 개뿐이고 나머지는 실행만 하면 됩니다. 학습은 세 번 돌아가며 한 번에 30~60초 정도 걸립니다. 덱 「Lab. Predict, Run, Complete, Compare」·「What We Now Understand About a VAE」.

## 0. Setup / run as is
Three cells: libraries, the MNIST images, and a look at the data. No blanks here.

**A note on the figures.** Every plot in this notebook is labelled in English, because Colab has no Korean font installed by default and Korean text inside a figure would come out as empty boxes. The Korean explanation of each figure is in the markdown and in the `print` output around it.

> 요약: 라이브러리, MNIST 이미지, 데이터 확인 세 셀입니다. 그림 안의 글자는 전부 영문입니다. Colab에 한글 폰트가 없어서 그림 안에 한글을 넣으면 네모로 깨지기 때문입니다. 그림에 대한 한국어 설명은 그림 위아래의 마크다운과 `print` 출력에 있습니다.

In [ ]:
# RUN AS IS. 이 셀은 실행만 하면 됩니다. 라이브러리를 부르고 난수 씨앗을 고정합니다.
# Run as is. This only imports libraries and fixes the random seeds.
import math, os, gzip, struct, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(2)      # Colab CPU 기준 / matches the usual Colab CPU

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리를 채우고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막히면 바로 아래 Solution / 정답 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace the None above and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

print('torch', torch.__version__, '| numpy', np.__version__)
print('실행 장치 / device: cpu  (GPU 없이 충분합니다 / no GPU needed today)')

In [ ]:
# RUN AS IS. MNIST 원본 파일 네 개를 내려받아 배열로 바꿉니다. 코드를 읽을 필요는 없습니다.
# Run as is. Downloads the four raw MNIST files and turns them into arrays.
MNIST_FILES = ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz',
               't10k-images-idx3-ubyte.gz',  't10k-labels-idx1-ubyte.gz']
MIRRORS = ['https://ossci-datasets.s3.amazonaws.com/mnist/',
           'https://storage.googleapis.com/cvdf-datasets/mnist/',
           'https://raw.githubusercontent.com/fgnt/mnist/master/']

def fetch(name):
    """첫 번째로 응답하는 미러에서 내려받습니다 / download from the first mirror that answers."""
    if os.path.exists(name):
        return name
    errors = []
    for base in MIRRORS:
        try:
            urllib.request.urlretrieve(base + name, name)
            return name
        except Exception as exc:
            errors.append(f'{base}: {type(exc).__name__}')
    raise RuntimeError('MNIST 내려받기 실패 / could not download ' + name + '\n  ' + '\n  '.join(errors))

def idx_images(path):
    with gzip.open(path, 'rb') as f:
        _, n, r, c = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), np.uint8).reshape(n, r * c)

def idx_labels(path):
    with gzip.open(path, 'rb') as f:
        struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), np.uint8)

for _f in MNIST_FILES:
    fetch(_f)

# 픽셀을 0~1 로 맞춥니다 / scale the pixels to the range 0 to 1
X_TRAIN = torch.from_numpy(idx_images('train-images-idx3-ubyte.gz').astype(np.float32) / 255.0)
X_TEST  = torch.from_numpy(idx_images('t10k-images-idx3-ubyte.gz').astype(np.float32) / 255.0)
Y_TEST  = idx_labels('t10k-labels-idx1-ubyte.gz')
N_TRAIN, D_PIXELS = X_TRAIN.shape

print('학습 이미지 / training images:', tuple(X_TRAIN.shape))
print('시험 이미지 / test images:    ', tuple(X_TEST.shape))
print('픽셀 값 범위 / pixel range:    %.1f ~ %.1f' % (X_TRAIN.min(), X_TRAIN.max()))
# 기대값 / Expected  (60000, 784) / (10000, 784) / 0.0 ~ 1.0

In [ ]:
# RUN AS IS. 데이터가 어떻게 생겼는지 한 번만 봅니다.
# Run as is. One look at the data.
def show_row(ax_row, imgs, label=None):
    """한 줄에 이미지 여러 장을 그립니다 / draw a row of images."""
    for ax, img in zip(ax_row, imgs):
        ax.imshow(np.asarray(img).reshape(28, 28), cmap='gray_r', vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
    if label is not None:
        ax_row[0].set_ylabel(label, rotation=0, ha='right', va='center', fontsize=10)

fig, axes = plt.subplots(1, 8, figsize=(9, 1.4))
show_row(axes, X_TEST[:8])
fig.suptitle('MNIST, 28 x 28 grey images', y=1.12, fontsize=11)
plt.show()

print('한 장은 784 개의 숫자입니다. 오늘 이 784 차원을 몇 개의 숫자로 줄였다가 되살립니다.')
print('One image is 784 numbers. Today we squeeze those 784 into a few and bring them back.')

## 1. Predict / 예측
**Slides: Lab. Predict, Run, Complete, Compare**

Answer these three before running anything below. One line each is enough. **A wrong prediction is useful; a blank one is not.** You will read your own answers again in §5 and compare them with what actually happened.

1. **Does generation use the encoder?** To make a digit that is not a copy of any input image, does the network need to look at an image first?
2. **What would change if the KL term were removed from the loss?** The reconstructions? The codes z? The images generated from the prior? All three? None?
3. **How will prior samples differ from reconstructions?** Both come out of the same decoder. Will they look the same?

Keep in mind the two draws the deck separated:

    training     z ~ q(z | x)     the encoder looked at this image
    generation   z ~ p(z)         nothing was looked at; the prior is a fixed N(0, I)

> 요약: 아래를 실행하기 전에 세 가지를 한 줄씩 예측합니다. (1) 생성에 인코더를 쓰나요? (2) 손실에서 KL 항을 빼면 무엇이 달라질까요? 복원인가요, 코드 z 인가요, prior에서 생성한 이미지인가요? (3) prior 표본과 복원 이미지는 어떻게 다를까요? 예측이 틀려도 괜찮습니다. 비워 두는 것만 안 됩니다. §5에서 자기 답을 다시 읽고 결과와 비교합니다.

In [ ]:
# 여기는 빈칸 문제가 아닙니다. 정답 셀도 없습니다. 따옴표 사이에 여러분의 예측을 적으세요.
# This is not a coding exercise and it has no Solution cell. Put your own prediction between the quotes.
PREDICT_1 = ""   # 생성에 인코더가 필요한가? / Does generation need the encoder?
PREDICT_2 = ""   # KL 항을 빼면 무엇이 달라지나? / What changes if the KL term is removed?
PREDICT_3 = ""   # prior 표본과 복원은 어떻게 다른가? / How will prior samples differ from reconstructions?

PREDICTIONS = {'1. generation and the encoder': PREDICT_1,
               '2. removing the KL term':       PREDICT_2,
               '3. prior samples vs reconstructions': PREDICT_3}

for _k, _v in PREDICTIONS.items():
    print(f'{_k:38s} {_v if _v else "<비어 있음 / still empty>"}')
if not all(PREDICTIONS.values()):
    print('\n세 칸을 다 채우고 다시 실행하세요. 틀려도 괜찮습니다. 여기서 멈추지는 않습니다.')
    print('Fill all three and re-run. Wrong answers are fine. Nothing is blocked by this cell.')

## 2. Run / 실행
**Slides: Two Ways to Use the Same Decoder · Only Reconstruction Needs an Input**

Two cells define the network and train it. Read the picture of the network below, not the code.

    x  (784)  →  encoder  →  mu(x), logvar(x)      q(z | x) = N( mu(x), sigma(x)^2 )
                              ↓ draw one z
                 decoder  →  784 logits            p(x | z), one Bernoulli per pixel

The encoder does not return a code. It returns **a mean and a spread**, which is what "a distribution over z" means. One z is then drawn from that distribution, and only that z reaches the decoder.

The loss per image is

    loss  =  reconstruction term  +  KL term
          =  −log p(x | z) for the drawn z   +   KL( q(z | x) ‖ p(z) )

which is a **one-sample estimate of the negative ELBO** from Week 3. Minimizing it is maximizing the ELBO. Week 3 wrote the ELBO as reconstruction minus KL; flip the sign and it becomes something a minimizer can be pointed at.

**Today the library writes the two lines you will write yourself in §3.** `Normal(...).rsample()` draws z in a way that lets gradients pass back to `mu` and `logvar`, and `kl_divergence(q, p)` computes the KL between two Gaussians. They are here so that you can see the behaviour first, before writing anything. §3 opens them up.

> 요약: 인코더는 코드 하나가 아니라 평균과 퍼짐을 돌려줍니다. 그것이 "z 위의 분포"라는 말의 뜻입니다. 거기서 z 를 한 번 뽑아 디코더에 넣습니다. 손실은 복원항 + KL 항이고, 이것이 지난주 ELBO에 마이너스를 붙인 값의 한 표본 추정입니다. 오늘 §3에서 직접 쓸 두 줄을 이 절에서는 라이브러리가 대신 해 줍니다. 먼저 결과를 보고 나서 코드를 쓰기 위해서입니다.

In [ ]:
# RUN AS IS. 신경망의 모양입니다. 층의 크기는 오늘의 주제가 아닙니다.
# Run as is. The shape of the network. The layer sizes are not today's topic.
class VAE(nn.Module):
    def __init__(self, latent):
        super().__init__()
        self.latent = latent
        # 인코더 몸통 / encoder body: 784 -> 512 -> 256
        self.enc = nn.Sequential(nn.Linear(784, 512), nn.ReLU(),
                                 nn.Linear(512, 256), nn.ReLU())
        # 머리 둘 / two heads: q(z|x) 의 평균과 로그분산 / the mean and the log-variance of q(z|x)
        self.head_mu     = nn.Linear(256, latent)
        self.head_logvar = nn.Linear(256, latent)
        # 디코더 / decoder: latent -> 256 -> 512 -> 784 logits
        self.dec = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(),
                                 nn.Linear(256, 512), nn.ReLU(),
                                 nn.Linear(512, 784))

    def encode(self, x):
        """이미지를 받아 q(z|x) 의 평균과 로그분산을 돌려줍니다 / image -> mean and log-variance of q(z|x)."""
        h = self.enc(x)
        return self.head_mu(h), self.head_logvar(h)

    def decode(self, z):
        """코드를 받아 784 개의 logit 을 돌려줍니다 / code -> 784 logits, one per pixel."""
        return self.dec(z)

def decode_to_images(model, z):
    """logit 을 0~1 픽셀 값으로 바꿉니다 / turn logits into pixel values in 0..1."""
    with torch.no_grad():
        return torch.sigmoid(model.decode(torch.as_tensor(z, dtype=torch.float32))).numpy()

print('VAE 정의 완료 / VAE defined. 잠재 차원은 만들 때 정합니다 / the latent size is chosen at build time.')

In [ ]:
# RUN AS IS. 학습 반복문입니다. 오늘 이해할 것은 loss 세 줄이고 나머지는 표준 PyTorch 절차입니다.
# Run as is. A standard PyTorch loop. The three lines that matter are the loss lines.
def train_with_library(latent, epochs=8, batch=256, seed=SEED, kl_weight=1.0):
    torch.manual_seed(seed)
    model = VAE(latent)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    print(f'{"epoch":>5} {"loss":>9} {"reconstruction":>15} {"KL":>8}')
    for ep in range(1, epochs + 1):
        perm = torch.randperm(N_TRAIN)
        tot = rec_tot = kl_tot = 0.0
        for i in range(0, N_TRAIN, batch):
            xb = X_TRAIN[perm[i:i + batch]]
            mu, logvar = model.encode(xb)
            q = Normal(mu, torch.exp(0.5 * logvar))            # q(z|x), 인코더가 돌려준 분포
            z = q.rsample()                                    # z 를 한 개 뽑는다. 기울기가 통과하는 방식으로
            logits = model.decode(z)
            # 복원항. Bernoulli 디코더에서 -log p(x|z) 는 곧 이진 교차엔트로피입니다.
            # Reconstruction term. For a Bernoulli decoder, -log p(x|z) is binary cross-entropy.
            rec = F.binary_cross_entropy_with_logits(logits, xb, reduction='sum') / len(xb)
            # KL 항. q(z|x) 를 사전분포 p(z) = N(0, I) 와 비교합니다.
            # KL term. It compares q(z|x) against the prior p(z) = N(0, I).
            prior = Normal(torch.zeros_like(mu), torch.ones_like(mu))
            kl = kl_divergence(q, prior).sum() / len(xb)
            loss = rec + kl_weight * kl
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(xb); rec_tot += rec.item() * len(xb); kl_tot += kl.item() * len(xb)
        print(f'{ep:5d} {tot/N_TRAIN:9.2f} {rec_tot/N_TRAIN:15.2f} {kl_tot/N_TRAIN:8.2f}')
    return model

print('학습을 시작합니다. 30~60초 정도 걸립니다 / training now, about 30 to 60 seconds.')
vae16 = train_with_library(latent=16, epochs=8)
# 기대값 / Expected  loss 는 첫 에폭 200 부근에서 시작해 마지막에 105~115 사이로 내려갑니다.
#   The loss starts near 200 and ends somewhere between 105 and 115.
#   복원항은 내려가는데 KL 은 4 부근에서 18 부근까지 올라갑니다. 이상한 일이 아닙니다.
#   The reconstruction term falls while the KL rises from about 4 to about 18. That is not a bug.
#   숫자가 정확히 일치하지는 않습니다. 부동소수점 연산 순서가 기기마다 달라서입니다.
#   Your numbers will not match to the last digit; floating-point order differs between machines.
#   마지막 loss 가 200 근처에서 멈춰 있으면 학습이 안 된 것입니다.
#   A loss stuck near 200 means nothing was learned.

**Read the two columns before going on.** The reconstruction term falls the whole way, and the KL term *rises*. This is the Week 3 table again: the ELBO improves while the KL to the prior grows, because the two pieces are traded against each other, and the loss is their sum.

> 요약: 복원항은 계속 내려가는데 KL 항은 오히려 올라갑니다. 지난주 표에서 ELBO가 오르는데 prior와의 KL도 같이 커지던 것과 같은 현상입니다. 두 항은 서로 맞바꾸는 관계이고 손실은 그 합입니다. 덱 「The Identity in Numbers」.

## 2b. Observe / 관찰
The next cell draws three rows from the **same trained decoder**.

| Row | Where z came from | Was an image looked at? |
|---|---|---|
| input | — | the original image |
| reconstruction | z ~ q(z given x) | yes, this image |
| generation | z ~ p(z) = N(0, I) | no, nothing at all |

Write down what you see before reading the next markdown cell.

> 요약: 세 줄 모두 같은 디코더에서 나옵니다. 첫 줄은 원본, 둘째 줄은 그 이미지를 인코더에 넣어 얻은 q(z|x)에서 z를 뽑아 복원한 것, 셋째 줄은 아무 이미지도 보지 않고 prior N(0, I)에서 z를 뽑아 생성한 것입니다. 다음 마크다운을 읽기 전에 무엇이 보이는지 먼저 적으세요.

In [ ]:
# RUN AS IS. 세 줄을 그립니다. 그림 코드는 읽지 말고 그림만 보세요.
# Run as is. Do not read the plotting code; look at the picture.
torch.manual_seed(3)
idx = torch.randperm(len(X_TEST))[:8]
originals = X_TEST[idx]

with torch.no_grad():
    mu, logvar = vae16.encode(originals)
    z_from_q = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)     # z ~ q(z|x)
    recons = torch.sigmoid(vae16.decode(z_from_q)).numpy()
    z_from_prior = torch.randn(8, vae16.latent)                        # z ~ p(z) = N(0, I)
    samples = torch.sigmoid(vae16.decode(z_from_prior)).numpy()

fig, axes = plt.subplots(3, 8, figsize=(9, 3.9))
show_row(axes[0], originals,  'input\n')
show_row(axes[1], recons,     'reconstruction\nz ~ q(z|x)')
show_row(axes[2], samples,    'generation\nz ~ p(z)')
plt.tight_layout(); plt.show()

print('첫 줄과 둘째 줄은 같은 자리에서 같은 숫자여야 합니다. 인코더가 그 이미지를 봤기 때문입니다.')
print('Rows 1 and 2 should match digit for digit. The encoder looked at that exact image.')
print('셋째 줄은 원본과 아무 관계가 없습니다. 인코더를 아예 쓰지 않았습니다.')
print('Row 3 has nothing to do with the originals. The encoder was not used at all.')

**What to notice.**

* Row 2 is the same digit as row 1 in every column, drawn a little softer than the original. The encoder looked at that image, so the code carries its identity; the softening is what a 16-number code and an averaging decoder give back.
* Row 3 is made of digits that are not in row 1. Most are recognizable, some are between two digits. **No image was fed in.** The only input was eight draws from N(0, I).
* Both rows came out of the same decoder weights. The decoder does not know or care where z came from.

So the answer to prediction 1 is visible: generation ran `torch.randn` and the decoder, and never called `encode`. The encoder exists to *train* the decoder, not to generate.

> 요약: 둘째 줄은 칸마다 첫 줄과 같은 숫자이고 조금 부드럽습니다. 인코더가 그 이미지를 봤기 때문입니다. 셋째 줄은 원본에 없는 숫자들이고, 이미지를 하나도 넣지 않았습니다. 들어간 것은 N(0, I)에서 뽑은 여덟 개의 z뿐입니다. 두 줄 다 같은 디코더에서 나왔습니다. 디코더는 z가 어디서 왔는지 모릅니다. 예측 1의 답이 여기서 보입니다. 생성은 인코더를 부르지 않습니다. 인코더는 디코더를 **학습시키기 위해** 있습니다.

**Explain, in one line each, before §3.**
(1) Why is row 2 blurrier than row 1? (2) Row 3 used `torch.randn(8, 16)`. Which distribution is that, and why is it the *right* one to draw from?

*질문. (1) 둘째 줄은 왜 원본보다 흐린가요? (2) 셋째 줄은 `torch.randn(8, 16)`을 썼습니다. 그것은 어떤 분포이고, 왜 하필 그 분포에서 뽑는 것이 맞나요?*

## 3. Now your two lines / 직접 쓰는 두 줄
**Slides: Draw the Noise First, Then Calculate the Code · Two Decisions Behind the Code**

The library did two things for us in §2. Here they are, one per cell.

### 3a. `reparameterize` — drawing z so that gradients survive

The encoder gives `mu` and `logvar`. We need a z drawn from N(mu, sigma²) where sigma = exp(0.5 · logvar). Drawing it the obvious way breaks training: a draw is not a function of `mu` you can differentiate, so the reconstruction term sends no gradient back through that draw. (The KL term depends on `mu` and `logvar` directly, so in a full VAE the encoder is not completely frozen. What is lost is the signal that says how to encode *this image*.)

The trick is to move the randomness out of the way. Draw the noise **first**, from a fixed distribution that has no parameters in it:

    eps ~ N(0, 1)            a draw, no parameters involved
    z   = ??? (mu, sigma, eps)      ordinary arithmetic, so gradients flow through mu and sigma

You need a z that is (a) genuinely a draw from N(mu, sigma²) and (b) built from `mu`, `std` and `eps` with nothing but `+` and `*`. Shifting a N(0, 1) draw moves its mean; scaling it changes its spread. Put those two operations together in the right order.

> 요약: 인코더가 `mu` 와 `logvar` 를 줍니다. 여기서 N(mu, sigma²) 의 표본 z 를 얻어야 하는데, 그냥 뽑아 버리면 **복원항에서 출발한 기울기가 그 표본을 거쳐 인코더까지 가지 못합니다.** (KL 항은 `mu` 와 `logvar` 에 직접 의존하므로 인코더가 완전히 멈추는 것은 아닙니다. 잃는 것은 「이 이미지를 어떻게 인코딩할지」를 알려 주는 신호입니다.) 그래서 잡음을 먼저 뽑습니다. `eps` 는 N(0, 1) 에서 뽑은 값이고 여기에는 학습할 것이 없습니다. 그다음 `mu`, `std`, `eps` 를 덧셈과 곱셈만으로 조합해 z 를 만듭니다. N(0, 1) 표본을 평행이동하면 평균이 바뀌고, 곱하면 퍼짐이 바뀝니다. 두 연산을 맞는 순서로 붙이면 됩니다.

In [ ]:
# 이 셀에서 쓰는 이름 / names used in this cell
#   mu      : q(z|x) 의 평균. 크기 (batch, latent) / the mean of q(z|x), shape (batch, latent)
#   logvar  : q(z|x) 의 로그분산. 인코더가 분산 대신 로그분산을 내보내는 이유는
#             어떤 실수가 나와도 exp 를 거치면 양수가 되기 때문입니다.
#             the log-variance of q(z|x). The encoder emits a log so that any real number
#             becomes a positive variance after exp.
#   std     : 표준편차. sigma = exp(0.5 * logvar) 입니다. 분산의 제곱근이므로 0.5 가 붙습니다.
#             the standard deviation. sigma = exp(0.5 * logvar); the 0.5 is the square root.
#   eps     : N(0, 1) 에서 뽑은 잡음. mu, logvar 와 아무 관계가 없습니다.
#             noise drawn from N(0, 1). It knows nothing about mu or logvar.
#   z       : q(z|x) 에서 뽑은 표본   <- 여기가 빈칸 / the sample from q(z|x)   <- this is the blank
#
# 참고 / Reference   torch.randn_like(t)
#   t 와 같은 모양의 텐서를 N(0, 1) 에서 뽑아 돌려줍니다.
#   Returns a tensor shaped like t, filled with draws from N(0, 1).
#   예 / e.g.   torch.randn_like(torch.zeros(2, 3))  ->  (2, 3) 짜리 표준정규 난수
#
# 텐서끼리의 + 와 * 는 원소별로 계산됩니다 / + and * between tensors work element by element.

def reparameterize(mu, logvar):
    """q(z|x) = N(mu, exp(logvar)) 에서 표본 하나를 뽑습니다 / draw one sample from q(z|x)."""
    std = torch.exp(0.5 * logvar)   # 로그분산에서 표준편차로 / log-variance to standard deviation
    eps = torch.randn_like(std)     # 잡음을 먼저 뽑는다 / draw the noise first

    # TODO. mu, std, eps 를 + 와 * 만으로 묶어 N(mu, std**2) 의 표본을 만드세요. 한 줄입니다.
    #       평행이동은 평균을, 곱셈은 퍼짐을 바꿉니다. 어느 쪽을 먼저 해야 할까요?
    # TODO. Combine mu, std and eps with + and * into a draw from N(mu, std**2). One line.
    #       A shift moves the mean, a scale changes the spread. Which one applies to eps?
    z = None

    if z is None:
        raise NotImplementedError('z' + BLANK_MSG)
    return z

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 이 셀에서 쓰는 이름 / names used in this cell
#   mu      : q(z|x) 의 평균 / the mean of q(z|x)
#   logvar  : q(z|x) 의 로그분산 / the log-variance of q(z|x)
#   std     : 표준편차 sigma = exp(0.5 * logvar) / the standard deviation
#   eps     : N(0, 1) 에서 뽑은 잡음 / noise drawn from N(0, 1)
#   z       : q(z|x) 에서 뽑은 표본 / the sample from q(z|x)

def reparameterize(mu, logvar):
    """q(z|x) = N(mu, exp(logvar)) 에서 표본 하나를 뽑습니다 / draw one sample from q(z|x)."""
    std = torch.exp(0.5 * logvar)   # 로그분산에서 표준편차로 / log-variance to standard deviation
    eps = torch.randn_like(std)     # 잡음을 먼저 뽑는다 / draw the noise first

    z = mu + std * eps              # 빈칸이었던 줄. eps 를 std 로 늘리고 mu 만큼 옮긴다
                                    # the blank. Scale eps by std, then shift it by mu

    if z is None:
        raise NotImplementedError('z' + BLANK_MSG)
    return z

In [ ]:
# 직접 작성했다면 정답 셀을 건너뛰고 여기서 먼저 검사합니다.
# If you wrote the line yourself, skip the Solution cell and check here first.
# RUN AS IS. 검사 방법 두 가지입니다. (1) 많이 뽑아서 평균과 표준편차가 맞는지, (2) 기울기가 통하는지.
# Run as is. Two checks: the draws have the right mean and spread, and gradients reach mu and logvar.
torch.manual_seed(0)
_mu     = torch.tensor([[0.5, -1.0]])
_logvar = torch.tensor([[0.0, -2.0]])            # std = 1.0 과 0.368 / std = 1.0 and 0.368
_target_std = torch.exp(0.5 * _logvar)

_n = 200_000
_draws = reparameterize(_mu.expand(_n, 2), _logvar.expand(_n, 2))
print('표본 평균 / sample mean      ', np.round(_draws.mean(0).tolist(), 3), ' 기대 / expected [ 0.5   -1.   ]')
print('표본 표준편차 / sample sd    ', np.round(_draws.std(0).tolist(), 3),  ' 기대 / expected [ 1.    0.368]')
_err = max(abs(_draws.mean(0) - _mu[0]).max().item(), abs(_draws.std(0) - _target_std[0]).max().item())
print(f'가장 큰 오차 / largest deviation {_err:.4f}')
print('20만 표본이므로 몬테카를로 오차는 대략 sd/sqrt(n) = 0.002 규모입니다. 0.01 아래면 통과로 봅니다.')
print('With 200,000 draws the Monte Carlo error is about sd/sqrt(n) = 0.002, so under 0.01 is a pass.')
assert _err < 0.01, (
    '평균이나 퍼짐이 맞지 않습니다. z = mu 만 쓰면 퍼짐이 0, z = eps 만 쓰면 평균이 0 이 됩니다. '
    'z = mu + std + eps 처럼 더하기만 하면 퍼짐이 1 로 고정됩니다. '
    '/ Wrong mean or spread. z = mu alone gives spread 0; z = eps alone gives mean 0; '
    'z = mu + std + eps gives a spread that is stuck at 1.')

torch.manual_seed(1)
_m = torch.tensor([[0.5, -1.0]], requires_grad=True)
_l = torch.tensor([[0.0, -2.0]], requires_grad=True)
reparameterize(_m, _l).sum().backward()
print('\nd z / d mu     ', _m.grad.tolist(), ' 기대 / expected [[1.0, 1.0]]')
print('d z / d logvar ', np.round(_l.grad.tolist(), 4), ' 0 이 아니어야 합니다 / must not be zero')
assert torch.allclose(_m.grad, torch.ones_like(_m)) and _l.grad.abs().min() > 0, (
    '기울기가 이 표본을 거쳐 가지 않습니다. torch.normal(mu, std) 를 쓰면 mu 와 logvar 에 정확히 0 이 돌아오고 '
    '복원항은 인코더에 아무 신호도 보내지 못합니다. .detach() 를 붙여도 같습니다. '
    '/ No gradient reaches mu or logvar through this draw. torch.normal(mu, std) returns exactly zero, '
    'so the reconstruction term sends the encoder no signal. A stray .detach() does the same.')
print('\nOK. 표본도 맞고 기울기도 통과합니다 / the draws are right and the gradients flow.')

### 3b. `kl_term` — which distribution is q compared against?

Week 3's rule: a KL is only defined once you name **both** distributions. Inside the ELBO there is exactly one KL, and the whole question is what sits on its right-hand side.

Three different distributions have been on the board, and they are not interchangeable:

| | what it is | available during training? |
|---|---|---|
| q(z given x) | what the encoder returned for this image | yes, we just computed it |
| p(z given x) | the true posterior | no. This is the gap from Week 3, and it is not computable |
| p(z) | the prior, a fixed N(0, I) with nothing learned in it | yes, it is a constant |

The KL term in the loss pulls q(z given x) toward one of these. Pick the one the ELBO actually contains — and note that it has to be one we can compute, or there would be nothing to train on.

**Why this term is what makes generation work.** Generation draws z from p(z). Training draws z from q(z given x). If those two live in different places, the decoder is asked at generation time about a region it was never trained on. This KL is the term that keeps them compatible. §4 removes it and looks at the damage.

> 요약: KL은 두 분포를 지정해야 정의됩니다. 후보는 셋입니다. q(z|x)는 인코더가 이 이미지에 대해 돌려준 분포, p(z|x)는 참 사후분포로 계산할 수 없는 것(지난주의 간극), p(z)는 학습할 것이 없는 고정된 N(0, I)입니다. ELBO 안의 KL이 q(z|x)를 끌어당기는 대상은 계산 가능한 쪽이어야 합니다. 이 항이 있어야 생성이 됩니다. 학습은 q(z|x)에서, 생성은 p(z)에서 z를 뽑는데 둘이 딴 데 있으면 디코더는 학습한 적 없는 자리를 질문받게 됩니다. §4에서 이 항을 빼고 무슨 일이 벌어지는지 봅니다.

In [ ]:
# 이 셀에서 쓰는 이름 / names used in this cell
#   mu, logvar : 인코더가 이 이미지에 대해 돌려준 값 / what the encoder returned for this image
#   q          : q(z|x) = N(mu, exp(logvar)). 위 두 값으로 이미 만들어 두었습니다.
#                already built from mu and logvar above
#   target     : q 가 비교되는 상대 분포   <- 여기가 빈칸
#                the distribution q is compared against   <- this is the blank
#
# 참고 / Reference   Normal(mean_tensor, sd_tensor)
#   평균과 표준편차를 텐서로 받아 정규분포를 만듭니다. 분산이 아니라 표준편차입니다.
#   Builds a normal distribution from a mean and a standard deviation, not a variance.
#   예 / e.g.   Normal(torch.full_like(mu, 3.0), torch.full_like(mu, 2.0))  ->  모든 차원에서 N(3, 4)
#
# 참고 / Reference   torch.zeros_like(t) / torch.ones_like(t)
#   t 와 같은 모양의, 전부 0 또는 전부 1 인 텐서를 만듭니다.
#   Tensors shaped like t, filled with 0 or with 1.
#
# 참고 / Reference   kl_divergence(a, b)
#   KL( a || b ) 를 차원마다 닫힌꼴로 돌려줍니다. 앞의 인자가 왼쪽 분포입니다.
#   Returns KL( a || b ) per dimension in closed form. The first argument is the left distribution.

def kl_term(mu, logvar):
    """KL( q(z|x) || ? ) 를 이미지마다 하나의 수로 돌려줍니다 / one number per image."""
    q = Normal(mu, torch.exp(0.5 * logvar))   # 인코더가 돌려준 분포 / the distribution the encoder returned

    # TODO. ELBO 안의 KL 에서 q 는 어느 분포와 비교되나요? 그 분포를 한 줄로 만드세요.
    #       평균과 표준편차 모두 mu 와 같은 모양의 텐서로 넣어야 합니다.
    # TODO. Inside the ELBO, which distribution is q compared against? Build it in one line.
    #       Both its mean and its standard deviation must be tensors shaped like mu.
    target = None

    if target is None:
        raise NotImplementedError('target' + BLANK_MSG)
    return kl_divergence(q, target).sum(dim=1)   # 잠재 차원에 대해 더한다 / sum over latent dimensions

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 이 셀에서 쓰는 이름 / names used in this cell
#   mu, logvar : 인코더가 이 이미지에 대해 돌려준 값 / what the encoder returned for this image
#   q          : q(z|x) = N(mu, exp(logvar))
#   target     : q 가 비교되는 상대 분포 / the distribution q is compared against

def kl_term(mu, logvar):
    """KL( q(z|x) || ? ) 를 이미지마다 하나의 수로 돌려줍니다 / one number per image."""
    q = Normal(mu, torch.exp(0.5 * logvar))   # 인코더가 돌려준 분포 / the distribution the encoder returned

    target = Normal(torch.zeros_like(mu), torch.ones_like(mu))
                                              # 빈칸이었던 줄. 사전분포 p(z) = N(0, I) 입니다
                                              # the blank. The prior p(z) = N(0, I)

    if target is None:
        raise NotImplementedError('target' + BLANK_MSG)
    return kl_divergence(q, target).sum(dim=1)   # 잠재 차원에 대해 더한다 / sum over latent dimensions

In [ ]:
# 직접 작성했다면 정답 셀을 건너뛰고 여기서 먼저 검사합니다.
# If you wrote the line yourself, skip the Solution cell and check here first.
# RUN AS IS. 세 가지를 확인합니다. q = p(z) 일 때 0 인지, §2 의 라이브러리 값과 같은지, 부호가 맞는지.
# Run as is. Three checks: zero when q equals the prior, agreement with §2, and the sign.
_mu0 = torch.zeros(4, 3); _lv0 = torch.zeros(4, 3)        # q = N(0, 1) = p(z)
print('q 가 사전분포와 같을 때 / when q equals the prior:', kl_term(_mu0, _lv0).tolist())
print('기대값 / Expected  [0.0, 0.0, 0.0, 0.0].  KL 은 두 분포가 같을 때만 0 입니다.')
assert torch.allclose(kl_term(_mu0, _lv0), torch.zeros(4), atol=1e-6), (
    'q 가 곧 사전분포인데 0 이 아닙니다. 비교 대상을 N(0, 1) 이 아닌 다른 것으로 두었을 가능성이 큽니다. '
    '/ q is exactly the prior, so this must be 0. The comparison target is probably not N(0, 1).')

with torch.no_grad():
    _m, _l = vae16.encode(X_TEST[:256])
    _mine = kl_term(_m, _l)
    _lib = kl_divergence(Normal(_m, torch.exp(0.5 * _l)),
                         Normal(torch.zeros_like(_m), torch.ones_like(_m))).sum(dim=1)
print(f'\n§2 학습된 인코더로 256 장 / on 256 images with the §2 encoder')
print(f'  여러분의 값 평균 / your mean  {_mine.mean():.4f}')
print(f'  §2 가 쓰던 값 평균 / the mean §2 used  {_lib.mean():.4f}')
print(f'  가장 큰 차이 / largest difference {(_mine - _lib).abs().max():.2e}   기대 / expected 0')
assert torch.allclose(_mine, _lib, atol=1e-5), (
    '§2 에서 쓰던 KL 과 다릅니다. 비교 대상 분포를 다시 보세요. '
    '/ This is not the KL §2 used. Look again at which distribution q is compared against.')
assert (_mine >= -1e-6).all(), 'KL 은 음수가 될 수 없습니다 / a KL cannot be negative.'
print('\nOK. 두 줄 다 준비되었습니다 / both of your lines are ready.')

## 4. Your code, and the KL term on and off
**Slides: What Changes When We Remove the KL Term? · Better Reconstruction, Larger Prior Mismatch**

The loop below is the §2 loop with the library's two calls replaced by **your** `reparameterize` and `kl_term`. Nothing else changed.

We train it twice, with a latent size of **2** so that every code can be drawn on a flat page:

    kl_weight = 1.0      the real loss.        loss = reconstruction + KL
    kl_weight = 0.0      the KL term removed.  loss = reconstruction only

The second one keeps the same stochastic encoder and noise draw but drops the KL term, so nothing ties its codes to the prior. It is not literally Week 2's deterministic autoencoder, but it is in Week 2's situation, and Week 2 ended on exactly this question: reconstruction was fine, but nothing told us where to draw new codes from.

**Before running: read your `PREDICT_2` again.** Which of the two will reconstruct better? Which will generate better? They are not the same question.

> 요약: 아래 반복문은 §2 와 같고 라이브러리 호출 두 개만 여러분이 쓴 `reparameterize` 와 `kl_term` 으로 바뀐 것입니다. 잠재 차원을 2 로 두어 코드 전체를 평면에 그릴 수 있게 하고, KL 가중치를 1 과 0 으로 두 번 학습합니다. 가중치 0 은 KL 항을 통째로 뺀 것입니다. 구조는 확률적 인코더와 잡음 그대로라 2주차의 결정적 오토인코더가 되는 것은 아니지만, 코드를 사전분포에 묶어 두는 항이 없다는 점에서 2주차와 같은 처지입니다. 실행 전에 `PREDICT_2` 를 다시 읽으세요. 둘 중 어느 쪽이 복원을 더 잘할까요. 어느 쪽이 생성을 더 잘할까요. 이 둘은 같은 질문이 아닙니다.

In [ ]:
# RUN AS IS. §2 의 반복문과 같고, 표시한 두 줄만 여러분의 함수로 바뀌었습니다.
# Run as is. The §2 loop, with the two marked lines now calling your functions.
def train_with_your_code(latent, kl_weight, epochs=6, batch=256, seed=SEED, tag=''):
    torch.manual_seed(seed)
    model = VAE(latent)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    print(f'--- latent {latent}, kl_weight {kl_weight}  {tag}')
    print(f'{"epoch":>5} {"loss":>9} {"reconstruction":>15} {"KL":>8}')
    for ep in range(1, epochs + 1):
        perm = torch.randperm(N_TRAIN)
        rec_tot = kl_tot = 0.0
        for i in range(0, N_TRAIN, batch):
            xb = X_TRAIN[perm[i:i + batch]]
            mu, logvar = model.encode(xb)
            z = reparameterize(mu, logvar)                                   # <- 여러분의 함수 / yours
            rec = F.binary_cross_entropy_with_logits(model.decode(z), xb, reduction='sum') / len(xb)
            kl = kl_term(mu, logvar).mean()                                  # <- 여러분의 함수 / yours
            loss = rec + kl_weight * kl
            opt.zero_grad(); loss.backward(); opt.step()
            rec_tot += rec.item() * len(xb); kl_tot += kl.item() * len(xb)
        rec_ep, kl_ep = rec_tot / N_TRAIN, kl_tot / N_TRAIN
        print(f'{ep:5d} {rec_ep + kl_weight * kl_ep:9.2f} {rec_ep:15.2f} {kl_ep:8.2f}')
    return model

print('두 번 학습합니다. 합쳐서 1~2분 정도 걸립니다 / two trainings, one to two minutes in total.\n')
vae2 = train_with_your_code(2, kl_weight=1.0, tag='(the VAE)')
print()
ae2  = train_with_your_code(2, kl_weight=0.0, tag='(the KL term removed)')
# 기대값 / Expected
#   KL 가중치 1: 복원항이 145~155 사이로 내려오고 KL 은 6 근처에서 평평해집니다.
#     kl_weight 1: the reconstruction term settles between 145 and 155, and the KL flattens near 6.
#   KL 가중치 0: 복원항이 조금 더 낮게 내려가고, KL 열의 값은 에폭마다 계속 커집니다.
#     kl_weight 0: the reconstruction term goes a bit lower, and the KL column keeps growing.
#     가중치가 0 이라 손실에 들어가지 않을 뿐, 값은 계속 계산해서 보여 주고 있습니다.
#     The KL is still computed and printed; it is simply not part of the loss.
#   숫자가 마지막 자리까지 같지는 않습니다 / your numbers will not match to the last digit.

### 4b. Observe / 관찰
The next figure has two columns, one per model. Each column shows

* **top:** where the encoder put the 3,000 test images, as dots, one dot per image, coloured by digit. The dashed circles are radius 1 and radius 2 of the prior p(z) = N(0, I), so almost all of the prior's mass is inside them.
* **bottom:** eight images decoded from **the same eight draws of z ~ p(z)** in both columns.

**Look at the numbers on the axes before you compare the shapes.** The two scatter plots are not drawn to the same scale.

> 요약: 왼쪽은 KL 항이 있는 모형, 오른쪽은 뺀 모형입니다. 위쪽은 인코더가 시험 이미지 3,000장을 어디에 놓았는지를 점으로 찍은 것이고, 점선 원은 사전분포 p(z) = N(0, I) 의 반지름 1과 2입니다. 아래쪽은 두 열 모두 **같은 여덟 개의 z ~ p(z)** 를 디코딩한 것입니다. 모양을 비교하기 전에 두 산점도의 축 눈금 숫자를 먼저 보세요. 두 그림은 같은 축척이 아닙니다.

In [ ]:
# RUN AS IS. 그림 코드입니다. 그림과 축 눈금만 보세요.
# Run as is. Look at the picture and at the axis numbers.
from matplotlib.patches import Circle

torch.manual_seed(2)
sub = torch.randperm(len(X_TEST))[:3000]
z_prior_common = torch.randn(8, 2)          # 두 모형에 같은 z 를 넣는다 / the same z for both models

fig = plt.figure(figsize=(10.5, 5.0))
gs = fig.add_gridspec(3, 2, height_ratios=[2.6, 0.42, 0.85], hspace=0.40, wspace=0.18)
for col, (model, title) in enumerate([(vae2, 'with the KL term  (the VAE)'),
                                      (ae2,  'without the KL term')]):
    with torch.no_grad():
        m, _ = model.encode(X_TEST[sub])
    m = m.numpy()
    ax = fig.add_subplot(gs[0, col])
    ax.scatter(m[:, 0], m[:, 1], c=Y_TEST[sub.numpy()], cmap='tab10', s=4, alpha=0.6, linewidths=0)
    for r in (1, 2):
        ax.add_patch(Circle((0, 0), r, fill=False, color='#2C2C2C', lw=1.2, ls='--'))
    lim = max(3.4, float(np.abs(m).max()) * 1.05)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_title(f'{title}\nwhere the encoder puts the images', fontsize=11, linespacing=1.5)
    ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')

    cap = fig.add_subplot(gs[1, col]); cap.axis('off')
    cap.text(0.5, 0.3, 'decoding the same eight draws of  z ~ p(z)', ha='center', fontsize=10.5,
             color='#8A6D00')
    inner = gs[2, col].subgridspec(1, 8, wspace=0.08)
    imgs = decode_to_images(model, z_prior_common)
    for k in range(8):
        a = fig.add_subplot(inner[0, k])
        a.imshow(imgs[k].reshape(28, 28), cmap='gray_r', vmin=0, vmax=1)
        a.set_xticks([]); a.set_yticks([])
plt.show()

Eight draws are a small sample. The next figure decodes a whole **grid** of z values covering the range the prior draws from, so nothing depends on which eight came up. Every image in each panel is an image that model can generate.

아래 표의 마지막 열은 인코더가 낸 **평균 `mu` 의 반지름**으로 셉니다(`|mu| < 3`). 표본 `z` 로 세어도 이 모형에서는 0.3 포인트 안에서 같습니다.

> 요약: 여덟 개는 표본이 너무 적습니다. 다음 그림은 사전분포가 뽑는 범위 전체를 격자로 훑어 디코딩합니다. 어떤 여덟 개가 나왔는지에 결과가 좌우되지 않습니다. 각 판에 있는 이미지는 모두 그 모형이 생성할 수 있는 이미지입니다.

In [ ]:
# RUN AS IS. 사전분포가 닿는 범위를 격자로 훑어 디코딩합니다.
# Run as is. Decode a grid of z values covering the range the prior draws from.
G = 8
grid = np.linspace(-2.2, 2.2, G)          # z ~ N(0, I) 표본의 약 97% 가 이 범위 안 / about 97% of draws
zz = np.array([[x, y] for y in grid[::-1] for x in grid], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 5.0))
for ax, (model, title) in zip(axes, [(vae2, 'with the KL term  (the VAE)'),
                                     (ae2,  'without the KL term')]):
    tiles = decode_to_images(model, zz).reshape(G, G, 28, 28)
    canvas = np.block([[tiles[r, c] for c in range(G)] for r in range(G)])
    ax.imshow(canvas, cmap='gray_r', vmin=0, vmax=1, extent=[-2.2, 2.2, -2.2, 2.2])
    ax.set_title(title + "\nwhat the prior's range decodes to", fontsize=11, linespacing=1.5)
    ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')
plt.tight_layout(); plt.show()

print('두 판에서 각각 몇 가지 숫자가 보이는지 세어 보세요. 없는 숫자가 있는지도 보세요.')
print('Count how many different digits you can find in each panel, and which ones are missing.')

In [ ]:
# RUN AS IS. 그림에서 본 것을 숫자로 확인합니다.
# Run as is. The same comparison as numbers.
R = 3.0                                    # 사전분포가 사실상 벗어나지 않는 반지름 / the prior effectively stays inside
P_INSIDE = 1 - math.exp(-R ** 2 / 2)       # 2차원 표준정규에서 |z| < R 일 확률 / exact for a 2-D standard normal

def summarize(model, name):
    with torch.no_grad():
        mu, logvar = model.encode(X_TEST)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        rec = F.binary_cross_entropy_with_logits(model.decode(z), X_TEST, reduction='sum') / len(X_TEST)
        kl = kl_term(mu, logvar).mean()
    inside = (mu.norm(dim=1) < R)
    counts = np.bincount(Y_TEST[inside.numpy()], minlength=10)
    print(f'{name:22s} {rec.item():9.2f} {kl.item():8.2f} {mu.abs().max().item():11.1f} '
          f'{inside.float().mean().item() * 100:14.1f}%')
    return counts

print('시험 자료 10,000장 기준. 마지막 열은 코드가 |mu| < 3 안에 있는 이미지의 비율입니다.')
print('On the 10,000 test images. The last column is the share whose code lies inside |mu| < 3.')
print(f'\n{"model":22s} {"rec":>9s} {"KL":>8s} {"max |mu|":>11s} {"inside |mu|<3":>15s}')
c1 = summarize(vae2, 'with the KL term')
c0 = summarize(ae2,  'without the KL term')
print(f'\n사전분포 p(z) 에서 뽑은 z 는 {P_INSIDE * 100:.1f}% 가 |z| < {R:.0f} 안에 들어옵니다.')
print(f'{P_INSIDE * 100:.1f}% of draws from the prior p(z) land inside |z| < {R:.0f}.')
print('그러니 마지막 열은 인코더 평균이 사전분포 표본이 주로 떨어지는 원 안에 있는 시험 이미지의 비율입니다. 생성 결과를 잰 것은 아닙니다(그 증거는 위 격자 그림).')
print('So the last column is the share of test images whose encoder mean sits where prior draws usually land. It does not measure what generation produces (the grid figure above is that evidence).')

print('\n|mu| < 3 안에 있는 시험 이미지의 숫자별 개수 / test images inside |mu| < 3, by digit')
print('digit   ' + ''.join(f'{d:>6d}' for d in range(10)))
print(f'{"with KL":8s}' + ''.join(f'{v:6d}' for v in c1))
print(f'{"no KL":8s}'  + ''.join(f'{v:6d}' for v in c0))
# 기대값 / Expected
#   KL 항이 있는 모형: |mu| < 3 비율이 90% 를 넘고, 열 개 숫자가 고르게 들어 있습니다.
#     with the KL term: over 90% inside, and all ten digits present in similar numbers.
#   KL 항이 없는 모형: 비율이 절반 아래로 떨어지고, 어떤 숫자는 거의 0 에 가깝습니다.
#     without it: the share drops below half, and some digits are close to zero.
#   복원항은 KL 항을 뺀 쪽이 보통 몇 nats 더 낮습니다. 여러분의 숫자로 확인하세요.
#     The reconstruction term is usually a few nats lower without the KL term. Check your own numbers.

**What to notice.**

* The scatter on the left sits on top of the prior circles. The right one has a similar shape, but its axes run about ten times further and the two dashed circles have shrunk to a speck at the origin. The codes are not "more spread out"; the prior is **looking at a small corner of them**.
* The last column of the table counts the test images whose encoder mean lies inside that corner, the radius-3 circle where 98.9% of prior draws land. With the KL term, over 90%. Without it, well under half.
* The per-digit counts say *which* images. On the right, one or two digit classes almost vanish from the prior's region, so those digits are unlikely to appear in samples. That last step is an inference, not a measurement: what the table measures is which codes lie in the region, and the direct evidence for what actually comes out is the grid figure above. Count the distinct digits in each panel. The right panel also looks ghostier, because a fixed grid step covers much more of that model's code space and lands between classes more often.
* And yet the reconstruction term is usually **lower** on the right. The model without the KL term is the better autoencoder and the worse generator. That is prediction 2, and it is the Week 2 question answered: good reconstruction does not by itself give you a rule for drawing new codes.

The KL term is not a regularizer bolted on for tidiness. It is the term that makes `z = torch.randn(...)` a sensible thing to hand the decoder.

> 요약: 왼쪽 산점도는 사전분포의 원 위에 겹쳐 있습니다. 오른쪽은 모양은 비슷해 보여도 축 눈금이 열 배 가까이 크고, 점선 원 두 개는 원점의 작은 점으로 줄어듭니다. 코드가 "더 퍼진" 것이 아니라 사전분포가 **그 코드들의 한쪽 구석만 보고 있는** 것입니다. 표의 마지막 열은 인코더 평균이 그 구석(사전분포 표본의 98.9% 가 떨어지는 반지름 3 인 원) 안에 있는 시험 이미지의 비율입니다. KL 항이 있으면 90%가 넘고, 없으면 절반 아래로 떨어집니다. 숫자별 개수를 보면 오른쪽에서는 특정 숫자가 사전분포 영역에서 거의 사라집니다. 그래서 그 숫자들은 생성에서 나올 가능성이 낮습니다. 다만 이 마지막 한 걸음은 추론이고 측정이 아닙니다. 표가 재는 것은 「어느 코드가 그 영역 안에 있는가」이고, 실제로 무엇이 나오는지에 대한 직접 증거는 위의 격자 그림입니다. 그런데도 복원항은 보통 오른쪽이 더 낮습니다. KL을 뺀 모형이 오토인코더로서는 더 낫고 생성기로서는 더 나쁩니다. 이것이 예측 2의 답이고 2주차 질문에 대한 답입니다. KL 항은 장식이 아니라 `z = torch.randn(...)` 을 디코더에 넣는 일을 말이 되게 만들어 주는 항입니다.

**Explain, in one line each.**
(1) Why does removing the KL term *improve* the reconstruction term? (2) The right-hand model also drew z with `reparameterize`, so its codes had noise added too. Why did that noise alone not keep them near the origin?

*질문. (1) KL 항을 빼면 복원항은 왜 오히려 좋아지나요? (2) 오른쪽 모형도 `reparameterize` 로 z를 뽑았으니 잡음은 똑같이 들어갔습니다. 그런데도 코드가 원점 근처에 머물지 않은 이유는 무엇인가요?*

## 5. Three lines + environment / in-class wrap-up
Write what you changed, the main number, and whether that number says the generative model improved. Not graded, nothing to submit; this only practices the format the assignments use.

### The four-week connection
* Week 2: train encoder and decoder weights for reconstruction. Nothing said where new codes should come from.
* Week 3: fix the toy's generative model and adjust q by hand for one observation. ELBO + gap = log p(x).
* Week 4: an encoder predicts q(z given x) for **every** image (amortized inference), and the decoder is trained with it by minimizing a one-sample estimate of the negative ELBO.
* Next: better decoders and better priors, with the same objective.

The word *amortized* is the whole difference between Week 3 and Week 4. Last week, one observation took a full optimization to find its q. This week a single forward pass of the encoder produces q for an image it has never seen. Learning how to produce q was paid for once, during training; each new image still needs one encoder forward pass.

> 요약: 2주차는 복원만 학습했고 새 코드를 어디서 뽑을지는 정하지 못했습니다. 3주차는 토이를 고정하고 관측 하나의 q를 손으로 맞췄습니다. 4주차는 인코더가 모든 이미지에 대해 q(z|x)를 예측하고, 디코더를 음의 ELBO(한 표본 추정)로 함께 학습합니다. 이것이 amortized inference 입니다. 지난주에는 관측 하나마다 최적화를 새로 돌렸지만, 이번 주에는 처음 보는 이미지도 인코더 한 번 통과로 q가 나옵니다. q 를 만드는 법을 학습 때 한 번에 배워 두었고, 새 이미지마다 인코더 한 번 통과는 여전히 필요합니다. 덱 「What We Now Understand About a VAE」.

In [ ]:
import sys, platform, datetime
print('Runtime:', platform.platform(), '| Python:', sys.version.split()[0])
print('torch:', torch.__version__, '| seed:', SEED)
print('Executed at:', datetime.datetime.now().astimezone().isoformat(timespec='minutes'))

print('\n예측과 실제 / your predictions, read again')
for _k, _v in PREDICTIONS.items():
    print(f'  {_k:38s} {_v if _v else "<비어 있음 / was left empty>"}')

summary = {'무엇을 바꿨는가 / What I changed': 'write here / 직접 작성',
           '핵심 수치 / Main number': 'write here / 직접 작성',
           '생성모형이 좋아졌다는 뜻인가 / Does it mean the generative model improved':
               'write here / 직접 작성'}
for _k, _v in summary.items():
    print(_k + ': ' + _v)

## Optional A. A straight line in latent space
Two images, their codes, and the codes in between. Nothing here is new machinery; it only shows that the region the KL term kept populated is connected, not a set of isolated islands.

> 요약: 이미지 두 장의 코드를 잇는 직선 위의 점들을 디코딩합니다. 새로운 장치는 없고, KL 항이 채워 둔 영역이 섬처럼 끊긴 곳이 아니라 이어진 곳이라는 점만 보여 줍니다.

In [ ]:
# RUN AS IS.
a_idx = int(np.where(Y_TEST == 0)[0][0])
b_idx = int(np.where(Y_TEST == 8)[0][0])
with torch.no_grad():
    za, _ = vae16.encode(X_TEST[a_idx:a_idx + 1])
    zb, _ = vae16.encode(X_TEST[b_idx:b_idx + 1])
path = torch.cat([(1 - t) * za + t * zb for t in torch.linspace(0, 1, 9)])
walk = decode_to_images(vae16, path)

fig, axes = plt.subplots(1, 9, figsize=(9.5, 1.3))
show_row(axes, walk)
fig.suptitle('from the code of a 0 to the code of an 8, in a straight line', y=1.16, fontsize=11)
plt.show()
print('양 끝은 두 이미지의 평균 코드이고, 가운데는 어떤 이미지의 코드도 아닙니다.')
print('The two ends are the mean codes of two images; the middle is nobody\'s code.')

## Optional B. What happens if you draw z the obvious way
`torch.normal(mu, std)` also returns a draw from N(mu, sigma²). The draws are fine. The gradient is what breaks: PyTorch returns exactly zero for both `mu` and `logvar`, so nothing reaches the encoder **through this draw**. Run it and look at the two rows. (In a full VAE the KL term still touches `mu` and `logvar` directly, so the encoder is not completely frozen. The reconstruction signal is what disappears, and that is the signal that says how to encode this particular image.)

> 요약: `torch.normal(mu, std)` 도 N(mu, sigma²) 의 표본을 돌려줍니다. 표본 자체는 문제가 없습니다. 망가지는 것은 기울기입니다. PyTorch가 `mu` 와 `logvar` 양쪽에 정확히 0 을 돌려주므로 **이 표본을 거쳐서는** 인코더에 아무 신호도 도착하지 않습니다. (전체 VAE 에서는 KL 항이 `mu`·`logvar` 에 직접 닿으므로 인코더가 완전히 멈추는 것은 아닙니다. 사라지는 것은 복원 신호, 즉 이 이미지를 어떻게 인코딩할지를 알려 주는 신호입니다.)

In [ ]:
# RUN AS IS.
for name, draw in [('reparameterize (mu + std * eps)', lambda m, l: reparameterize(m, l)),
                   ('torch.normal(mu, std)',           lambda m, l: torch.normal(m, torch.exp(0.5 * l)))]:
    m = torch.tensor([[0.5, -1.0]], requires_grad=True)
    l = torch.tensor([[0.0, -2.0]], requires_grad=True)
    draw(m, l).sum().backward()
    print(f'{name:34s} d/d mu {m.grad.tolist()}   d/d logvar {np.round(l.grad.tolist(), 4).tolist()}')
print('\n두 번째 줄의 기울기는 전부 0 입니다. 손실이 아무리 커도 이 표본을 거쳐서는 신호가 가지 않습니다.')
print('Every gradient on the second row is zero. However large the loss, nothing reaches mu or logvar through this draw.')

## Optional C. The KL in closed form
`kl_divergence` did the algebra for us. For q = N(mu, sigma²) and p = N(0, 1) the answer is short enough to write down:

    KL( q ‖ p )  =  ½ · Σ over dimensions ( mu² + sigma² − 1 − log sigma² )
                 =  −½ · Σ ( 1 + logvar − mu² − exp(logvar) )

Nothing to fill in; run it and confirm the two agree. Most VAE code you will read online writes the second line directly instead of building distribution objects.

> 요약: 앞에서 `kl_divergence` 가 대신 해 준 계산의 닫힌꼴입니다. 빈칸은 없고, 두 값이 같은지 확인만 하면 됩니다. 인터넷에서 보는 VAE 코드 대부분은 분포 객체를 만들지 않고 아래 둘째 줄을 그대로 씁니다. 지난주 노트북의 `kl_gauss` 에서 목표 분포를 N(0, 1) 로 둔 특수한 경우입니다.

In [ ]:
# RUN AS IS.
with torch.no_grad():
    mu, logvar = vae16.encode(X_TEST[:5])
closed_form = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).sum(dim=1)
print('kl_term 으로 / via kl_term      ', np.round(kl_term(mu, logvar).tolist(), 4))
print('닫힌꼴로 / closed form          ', np.round(closed_form.tolist(), 4))
print('가장 큰 차이 / largest difference', f'{(kl_term(mu, logvar) - closed_form).abs().max():.2e}')
# 기대값 / Expected  두 줄이 같은 다섯 개의 수, 차이는 1e-5 아래 / the same five numbers, difference below 1e-5

### References / 참고
Concepts: Kingma & Welling, *Auto-Encoding Variational Bayes*, arXiv 1312.6114, **§2.3 The SGVB estimator and AEVB algorithm**, **§2.4 The reparameterization trick**, and **Appendix B** (the Gaussian KL in closed form).
Rezende, Mohamed & Wierstra, *Stochastic Backpropagation and Approximate Inference in Deep Generative Models*, arXiv 1401.4082.

### Course textbook alignment / 교재 연결
Page numbers below were checked against the actual PDFs, not from memory.

- 주교재 Main: Tomczak, *Deep Generative Modeling*, 2nd ed. (Springer, 2024). This week's range is **§5.3.3–§5.3.5 (pp.98–106)**; the reparameterization trick is **§5.3.3.2 (p.100)**, and posterior collapse is **§5.3.6 (pp.107–108)**.
- 부교재 Supplementary: Bishop & Bishop, *Deep Learning: Foundations and Concepts* (Springer, 2024), **§19.2 Variational Autoencoders (p.569)**, **§19.2.1 Amortized inference (p.572)**, **§19.2.2 The reparameterization trick (p.574)**.
- 참고 Background: Goodfellow, Bengio & Courville, *Deep Learning* (MIT Press, 2016), **§20.10.3 Variational Autoencoders (p.696)**.

> 요약: 쪽수는 보유 PDF 실물과 대조한 값입니다. Tomczak은 강의계획서와 같은 **2판(2024)** 기준입니다. 1판(2022)을 가진 분은 5.3.3 대신 **4.3.3**, reparameterization trick은 **4.3.3.2 (p.64)** 를 보면 됩니다. Goodfellow 쪽수는 인쇄본 기준이라 deeplearningbook.org 공개본과 2~3쪽 차이가 납니다.